# 10 · Testing pyramid, CI, logging, and secret safety

Inspect and run the quality gates that turn demos into maintainable evidence.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Testing pyramid

Unit tests isolate numerical and data contracts. Integration tests cross the FastAPI boundary.
Safety tests scan tracked source patterns. Workflow smoke tests add a real MySQL service and a
scheduled full-profile run.

In [ ]:
tests = []
for path in sorted(Path("tests").rglob("test_*.py")):
    tests.append({"layer": path.parent.name, "file": path.name,
                  "tests": path.read_text().count("def test_")})
display(pd.DataFrame(tests))

## Run the local suite

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"],
                        capture_output=True, text=True)
print(result.stdout[-2000:])
assert result.returncode == 0, result.stderr

## Static quality gate

In [ ]:
result = subprocess.run([sys.executable, "-m", "ruff", "check", "src", "scripts", "tests"],
                        capture_output=True, text=True)
print(result.stdout or "Ruff passed")
assert result.returncode == 0, result.stderr

## CI is executable documentation

In [ ]:
print(Path(".github/workflows/lint-test.yml").read_text())
print(Path(".github/workflows/mysql-smoke.yml").read_text())

## Structured logging

Logs need time, level, event, and correlation identifiers. They must not include raw text bodies,
credentials, or customer identifiers. This fictional dataset still practices the same boundary.

In [ ]:
from rookie_lab.software.logging import get_logger
logger = get_logger("notebook")
logger.info("notebook_quality_gate run_id=safe-demo")

## Secret scan and false confidence

Pattern tests catch common mistakes but not every secret format or leaked value in Git history.
Repository rules, short-lived credentials, least privilege, rotation, and provider-side scanning are
still required for a real system.

In [ ]:
secret_test = Path("tests/safety/test_no_secrets.py").read_text()
print(secret_test)
assert ".env" in Path(".gitignore").read_text()

## Exercises

Write a failing test before a bug fix; add a property test for bounding boxes; distinguish a flaky
test from nondeterministic training; specify which checks block pull requests and which run nightly.